In [1]:
import os
import sys
sys.path.insert(
    0, os.path.abspath('../../')
)

import json
import yaml

from pathlib import Path
from rich.console import Console
from rich.table import Table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
root_dir = Path("../../").resolve()
print("Root directory:", root_dir)

Root directory: /home/hgkahng/Workspaces/soft-prompt


In [3]:
N_train = 120_000

## 1. Load Synthetic Data

In [38]:
load_dir = root_dir / "results/agnews/gemini-2.0-flash/soft/"
print("Model directory:", load_dir)
print(*os.listdir(load_dir), sep="\n")

Model directory: /home/hgkahng/Workspaces/soft-prompt/results/agnews/gemini-2.0-flash/soft
template.jsonl
embeddings
config.yaml
data.jsonl
template_formatted.txt


In [39]:
# Print configurations

with open(load_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

style = "yellow" if cfg['hard'] else "cyan"

table = Table(title="Configuration(s)")
table.add_column("Name", justify="right", style="white", no_wrap=True)
table.add_column("Value", justify="left", style=style)
_ = [table.add_row(k, str(v)) for k, v in cfg.items()]

console = Console()
console.print(table)

         Configuration(s)          
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃         Name ┃ Value            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│   batch_size │ 50               │
│          cot │ False            │
│         data │ agnews           │
│         hard │ False            │
│ log_interval │ 2                │
│   max_tokens │ None             │
│        model │ gemini-2.0-flash │
│  sample_size │ 280000           │
│     strategy │ very spiky       │
│  temperature │ 1.0              │
└──────────────┴──────────────────┘

In [40]:
with open(load_dir / "data.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(f"Data size: {len(data):,}")

Data size: 280,041


In [41]:
data[-1]

{'label': [0.0, 0.0, 0.414, 0.585],
 'text': 'Tech Firms Lead Market Surge as AI Development Fuels Investor Optimism; Analysts Predict Strong Earnings Growth'}

In [42]:
# labels
labels = [d['label'] for d in data]
labels = np.array(labels)
# valid_idx = [i for i, l in enumerate(labels) if len(l) == 4]
# valid_labels = [l for i, l in enumerate(labels) if i in valid_idx]
# labels = np.array(valid_labels)
print(labels.shape[0])

280041


In [43]:
soft_labels = labels.copy()
soft_labels = soft_labels / soft_labels.sum(axis=1, keepdims=True)
assert soft_labels.sum(axis=1).all()

hard_labels = np.argmax(soft_labels, axis=1)

print("Soft labels, shape:", soft_labels.shape)
print("Hard labels, shape:", hard_labels.shape)

Soft labels, shape: (280041, 4)
Hard labels, shape: (280041,)


In [44]:
# text
texts = [d['text'] for _, d in enumerate(data)]
#texts = [d['text'] for i, d in enumerate(data) if i in valid_idx]
len(texts)

280041

In [45]:
# embeddings
embeddings = np.load(
    load_dir / "embeddings/openai/text-embedding-3-small/data.npy"
)

#embeddings = embeddings[valid_idx].copy()
print(embeddings.shape)

(280041, 1536)


In [46]:
assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

280041


In [47]:
import numpy as np

def probs_to_one_hot(probs):
  """
  Converts a 2D NumPy array of probabilities to a one-hot encoded array.

  Args:
    probs: A 2D NumPy array where each row represents a data
                         point and each column represents a class probability.
                         Each row is expected to sum to 1.

  Returns:
    A 2D NumPy array with the same shape as probs_array,
    but one-hot encoded.
  """
  # Find the index of the maximum probability for each row
  max_indices = np.argmax(probs, axis=1)

  # Create an array of zeros with the same shape as the input
  one_hot_array = np.zeros_like(probs, dtype=int)

  # Set the appropriate elements to 1
  # For each row i, set the column at max_indices[i] to 1
  one_hot_array[np.arange(probs.shape[0]), max_indices] = 1

  return one_hot_array

In [48]:
%%time
rng = np.random.default_rng(42)
sample_idx = rng.permutation(embeddings.shape[0])[:N_train]

labels = labels[sample_idx]
labels = probs_to_one_hot(labels)
embeddings = embeddings[sample_idx]
texts = [t for i, t in enumerate(texts) if i in sample_idx]

assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

120000
CPU times: user 5.99 s, sys: 113 ms, total: 6.1 s
Wall time: 6.1 s


## 2. Diversity Metrics

In [49]:
from softprompt.metrics.diversity import (
    vocabulary_size,
    distinct_n,
    average_pairwise_similarity,
    average_pairwise_similarity_by_class,
    inter_sample_ngram_freq
)

In [50]:
vocab_size = vocabulary_size(texts)
print(f"Vocab: {vocab_size:>7,}")

Vocab:   9,544


In [51]:
distinct_2 = distinct_n(texts, n=2)
print(f"Distinct-2: {distinct_2:.4f}")

Distinct-2: 0.0336


In [52]:
n = 20_000  # kernel crashes with full size
embeddings = embeddings[:n]
labels = labels[:n]

In [53]:
%%time
inter_sim, intra_sim = average_pairwise_similarity_by_class(
    embeddings, labels
)
print(f"Inter-class APS: {inter_sim:.4f}\n",
      f"Intra-class APS: {intra_sim:.4f}")

Inter-class APS: 0.2223
 Intra-class APS: 0.4955
CPU times: user 31.9 s, sys: 10.5 s, total: 42.5 s
Wall time: 4.7 s


In [54]:
%%time
avg_ps = average_pairwise_similarity(embeddings)
print(f"Avg Pairwise Similarity: {avg_ps:.4f}")

Avg Pairwise Similarity: 0.2906
CPU times: user 38.5 s, sys: 14.6 s, total: 53.1 s
Wall time: 3.66 s
